# Bayes@N with Inspect AI

How often would a model answer a question from this three-question quiz correctly on a single attempt? We'll grade four attempts per question and use Scorio's Inspect extension to estimate the quiz's mean success probability and its posterior standard deviation (SD). A local mock model supplies all twelve answers. [Scorio's Inspect guide](https://github.com/mohsenhariri/scorio/blob/python-v0.2.4/docs/integrations/inspect.rst) · [Inspect extension listing PR](https://github.com/UKGovernmentBEIS/inspect_ai/pull/6430)

Use a Python 3.11+ Jupyter kernel. For a fresh environment, run the following in a notebook cell, restart the kernel, then run the tutorial.

```python
%pip install "scorio[inspect]==0.2.4" "inspect-ai==0.3.288" ipywidgets
```

## Define the quiz and the binary scorer

In Inspect, one **epoch** is one attempt at every selected question. Give each question a stable ID and declare the IDs and epoch count before the run. The grader will compare the model's answer with the target letter.

In [1]:
from inspect_ai import Task
from inspect_ai import eval as inspect_eval
from inspect_ai.dataset import Sample
from inspect_ai.scorer import match
from inspect_ai.solver import generate, system_message

from scorio.integrations.inspect import bayes_at_n, record_epochs, validate_bayes_at_n

samples = [
    Sample(
        id="planet",
        input="Which planet is known as the Red Planet?  A. Mars  B. Venus",
        target="A",
    ),
    Sample(
        id="hexagon",
        input="How many sides does a hexagon have?  A. Five  B. Six",
        target="B",
    ),
    Sample(
        id="current",
        input="Which unit measures electric current?  A. Ampere  B. Volt",
        target="A",
    ),
]
sample_ids = [sample.id for sample in samples]
epochs = 4

`match(location="exact")` returns `C` for a correct answer and `I` for an incorrect one. Wrap it in `record_epochs` to keep the epoch on each grade. Attach `bayes_at_n` to the task's metrics so it receives all four grades for each question.

In [2]:
task = Task(
    name="multiple-choice-quiz",
    dataset=samples,
    solver=[system_message("Reply with the letter A or B only."), generate()],
    scorer=record_epochs(match(location="exact")),
    metrics=[bayes_at_n(sample_ids=sample_ids, epochs=epochs)],
    epochs=epochs,
)

## Run the task and validate the finished log

The mock answers B once, then A for the rest of the quiz. One active sample at a time makes the sequence reproducible. The example reloads the saved log so the same validation cell also works with a log from an earlier run.

In [3]:
from tempfile import TemporaryDirectory

from inspect_ai.log import read_eval_log
from inspect_ai.model import ModelOutput, ModelUsage, get_model

answers = ["B"] + ["A"] * (len(samples) * epochs - 1)
outputs = []
for answer in answers:
    output = ModelOutput.from_content(model="mockllm", content=answer)
    # Supply mock token counts so the example can run without a tokenizer download.
    output.usage = ModelUsage(input_tokens=1, output_tokens=1, total_tokens=2)
    outputs.append(output)

model = get_model("mockllm/model", custom_outputs=outputs, memoize=False)
with TemporaryDirectory() as log_dir:
    run = inspect_eval(
        task, model=model, max_samples=1, log_dir=log_dir, display="none"
    )[0]
    log = read_eval_log(run.location)

Inspect can display `NaN` for a live metric while grades are still arriving. On the completed log, `validate_bayes_at_n` checks that every declared question has exactly one binary grade for every epoch. It raises an error for incomplete or invalid runs.

In [4]:
moments = validate_bayes_at_n(log, sample_ids=sample_ids, epochs=epochs)
print(f"Posterior mean  {moments['posterior_mean']:.6f}")
print(f"Posterior SD    {moments['posterior_sd']:.6f}")

Posterior mean  0.555556
Posterior SD    0.089087


## Trace the estimate back to the grades

Build Scorio's question-by-attempt matrix from the log. Look up each grade by its sample ID and epoch so the order in which samples finished has no effect. A 1 denotes a correct answer.

In [5]:
import numpy as np
from scorio.eval import bayes

grades = {(s.id, s.epoch): s.scores["record_epochs"].value for s in log.samples}
R = np.array(
    [[grades[q, epoch] == "C" for epoch in range(1, epochs + 1)] for q in sample_ids],
    dtype=int,
)
for question, row in zip(sample_ids, R):
    print(f"{question:<8} {row.tolist()}   {row.sum()} of {epochs} correct")

np.testing.assert_allclose(
    bayes(R), [moments["posterior_mean"], moments["posterior_sd"]]
)

planet   [0, 1, 1, 1]   3 of 4 correct
hexagon  [0, 0, 0, 0]   0 of 4 correct
current  [1, 1, 1, 1]   4 of 4 correct


With an independent uniform Beta(1, 1) prior for each question, three correct answers in four attempts give the planet question a posterior mean of $(3+1)/(4+2)=4/6$. The hexagon and current questions give $1/6$ and $5/6$. Their average is **55.56%**, compared with the observed $7/12$ correct. The posterior SD of **8.91 percentage points** describes uncertainty about the mean success probability on these three questions.

For your own run, pass your configured `provider/model` name to `inspect_eval` and point `log_dir` at a directory you want to keep. Draw independent answers with the same model, prompt, sampling settings and grader on every epoch. Use the IDs of the questions actually selected, and validate the full log before reporting the result.